# Practice 52 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — weekly retail sales

A retail chain tracks weekly revenue for a single store.

`shift(1)` moves values **down** by 1 row — row 0 becomes NaN, row 1 gets the original row 0 value, etc. Use it to bring last week's value into the current row.

1. Add `prev_revenue`: last week's revenue using `shift(1)`.
2. Add `change`: this week minus `prev_revenue`.
3. Add `pct_change`: percentage change, rounded to 1 decimal (multiply by 100).
4. Flag `is_down`: True where revenue fell vs the prior week.
5. Which week had the largest single-week drop? Use `.idxmin()` on `change`.

In [6]:
sales = pd.DataFrame({
    'week':    pd.date_range('2024-01-07', periods=8, freq='W'),
    'revenue': [12400, 13100, 11800, 14200, 13500, 15800, 14900, 16200],
})

# Your code here
sales['prev_revenue'] = sales['revenue'].shift(1)
sales['change'] = sales['revenue'] - sales['prev_revenue']
sales['pct_change'] = round(sales['change']/sales['prev_revenue'], 3)*100
sales['is_down'] = sales['revenue']< sales['prev_revenue']
sales['change'].idxmin()


2

In [7]:
sales

,week,revenue,prev_revenue,change,pct_change,is_down
0,2024-01-07,12400,NaN,NaN,NaN,False
1,2024-01-14,13100,12400.0,700.0,5.6,False
2,2024-01-21,11800,13100.0,-1300.0,-9.9,True
3,2024-01-28,14200,11800.0,2400.0,20.3,False
4,2024-02-04,13500,14200.0,-700.0,-4.9,True
5,2024-02-11,15800,13500.0,2300.0,17.0,False
6,2024-02-18,14900,15800.0,-900.0,-5.7,True
7,2024-02-25,16200,14900.0,1300.0,8.7,False


---
## Level 2 — monthly housing prices

Monthly median sale prices for a city over one year. No step-by-step — figure out the approach.

- `rolling_avg`: 3-month rolling mean of `price`.
- `mom_change`: month-over-month price change using `shift(1)`.
- `above_trend`: True where `price` exceeds `rolling_avg` by more than 2%.
- Use NumPy to find: mean price, std, and the month with the peak rolling average.

In [70]:
prices = pd.DataFrame({
    'month': pd.date_range('2023-01-01', periods=12, freq='MS'),
    'price': [485000, 492000, 501000, 498000, 515000, 528000,
              541000, 537000, 522000, 535000, 548000, 562000],
})

# Your code here
prices['rolling_avg'] = prices['price'].rolling(3).mean()
prices['prev_m'] = prices['price'].shift(1)
prices['mom_change'] = prices['price'] - prices['prev_m']
prices['above_trend'] = prices['price']> prices['rolling_avg']*1.02
print(np.mean(prices['price'] ))
print(np.std(prices['price']))
print(prices['month'][np.argmax(np.nan_to_num(prices['rolling_avg'], nan = -np.inf))])



522000.0
23097.618924902195
2023-12-01 00:00:00


In [67]:
prices['rolling_avg'].dropna()


2     492666.666667
3     497000.000000
4     504666.666667
5     513666.666667
6     528000.000000
7     535333.333333
8     533333.333333
9     531333.333333
10    535000.000000
11    548333.333333
Name: rolling_avg, dtype: float64

---
## Level 3 — multi-ticker stock returns

Daily closing prices for three tickers over 10 trading days. `date` is already the index.

Answer these questions — no prescribed steps, choose your own approach:

1. Compute daily **log returns** for each ticker: `np.log(price / price.shift(1))`. Store as a new DataFrame `returns`.
2. Which ticker had the highest **mean** daily return? Which had the highest **volatility** (std of log returns)?
3. Use `np.corrcoef` to build a 3×3 correlation matrix of daily returns. Which pair of tickers is most correlated?  
   Hint: `np.corrcoef` expects each row to be one variable — pass `returns.dropna().values.T`.
4. Compute a **3-day rolling volatility** (rolling std of log returns) for each ticker. Which ticker is most volatile in the final 3 days?

In [75]:
stocks = pd.DataFrame({
    'date': pd.date_range('2024-01-02', periods=10, freq='B'),
    'AAPL': [185.2, 186.1, 184.5, 187.3, 188.9, 187.5, 190.2, 192.1, 191.0, 193.5],
    'MSFT': [374.0, 376.5, 373.2, 379.1, 381.3, 379.8, 383.5, 385.2, 384.0, 387.1],
    'GOOG': [140.5, 141.2, 139.8, 142.3, 143.1, 142.0, 144.5, 145.8, 144.2, 146.3],
}).set_index('date')

# Your code here

stocksp = stocks.shift(1)
returns = np.log(stocks/stocksp)


In [76]:
returns.mean(0).idxmax()

'AAPL'

In [77]:
returns.std(0).idxmax()

'GOOG'

In [83]:
cors = np.corrcoef(returns.dropna().values.T)


In [84]:

cd = pd.DataFrame(cors.copy(), columns=returns.columns,
                  index = returns.columns)

cd

,AAPL,MSFT,GOOG
AAPL,1.000000,0.944360,0.983795
MSFT,0.944360,1.000000,0.939746
GOOG,0.983795,0.939746,1.000000


In [85]:
for i in range(3):
    for j in range(3):
        if j<=i:
            cors[i,j] = 0

cdm = pd.DataFrame(cors, columns=returns.columns,
                  index = returns.columns)
cdm.stack().idxmax()


('AAPL', 'GOOG')

In [54]:
for i in returns.columns:
    cname = i+'rv'
    returns[cname] = returns[i].rolling(3).std()

In [62]:
returns.iloc[-1][-3:].idxmax()

'GOOGrv'